# 01 — VQGAE Quickstart

Encode a few SMILES into the VQGAE fragment latent, then decode them back.
Uses only the public API and the pretrained checkpoints on
[Hugging Face](https://huggingface.co/tagirshin/VQGAE).

What you'll see:
- `encode_smiles` returns both the discrete codebook indices `(N, 51)` and
  the 4096-dim integer fragment-count histogram per molecule. The histogram
  is what optimizers (PyGAD, Optuna, …) search over in tutorials 2 & 3.
- `decode_population` runs the canonical three-step decode pipeline:
  histogram → fragment ordering (via `OrderingNetwork`) → atoms + bonds.
- Validity flags come from chemistry sanity (connected, valences OK,
  no banned substructures, ring sizes 4–8).


## Imports + checkpoints

In [ ]:
from huggingface_hub import hf_hub_download
from VQGAE import VQGAE, OrderingNetwork, encode_smiles, decode_population

REPO = "tagirshin/VQGAE"
vqgae_ckpt = hf_hub_download(REPO, "vqgae.ckpt")
onn_ckpt   = hf_hub_download(REPO, "ordering_network.ckpt")

## Load the model heads

The same `vqgae.ckpt` weights drive both `encode` and `decode`; only the
forward path differs. `OrderingNetwork` is a separate small model that
predicts canonical atom orderings.


In [ ]:
BATCH = 5
enc = VQGAE.load_from_checkpoint(vqgae_ckpt, task="encode", batch_size=BATCH, map_location="cpu").eval()
dec = VQGAE.load_from_checkpoint(vqgae_ckpt, task="decode", batch_size=BATCH, map_location="cpu").eval()
onn = OrderingNetwork.load_from_checkpoint(onn_ckpt, batch_size=BATCH, map_location="cpu").eval()

## Encode SMILES → fragment counts

**Where do these SMILES come from?** `encode_smiles` accepts any SMILES
string — lowercase aromatic (RDKit / OpenEye / web-drawer style), Kekulé,
with or without stereo, even quirky dialects. Internally it routes
through `VQGAE.smiles_to_mol`, which tries chython first and falls back
to RDKit canonicalisation if chython rejects the input.

If you only need the parsed `MoleculeContainer` (without encoding it),
call `smiles_to_mol(smi)` directly:

```python
from VQGAE import smiles_to_mol
mol = smiles_to_mol("CC(=O)Oc1ccccc1C(=O)O")
```

For an existing RDKit `Mol` object, use `rdkit_to_mol(rdkit_mol)`.

In [ ]:
SMILES_LIST = [
    "CC(=O)Oc1ccccc1C(=O)O",         # aspirin
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",  # caffeine
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",    # ibuprofen
    "OC(=O)C(N)Cc1ccccc1",           # phenylalanine
    "C1=CC=CC=C1",                   # benzene
]
codebook_inds, counts = encode_smiles(SMILES_LIST, enc, batch_size=BATCH)
print("codebook_inds:", codebook_inds.shape, "frag_counts:", counts.shape)
print("atom totals per molecule:", counts.sum(-1).tolist())

## Decode the histogram back (round-trip)

In [ ]:
mols, validity, ordering_scores = decode_population(counts, dec, onn, batch_size=BATCH)
for orig, mol, ok, sc in zip(SMILES_LIST, mols, validity, ordering_scores):
    rec = str(mol) if ok else "(invalid)"
    print(f"{orig:35s} -> {rec:35s}  ord={sc:.3f}")
print(f"\n{sum(validity)}/{len(validity)} valid")

> **Note on validity**: round-trip on real drug-like molecules is typically
> 60–90% valid; on the wider distribution of GA/BO-proposed candidates it
> drops to 10–30%. Filtering happens downstream — see tutorial 2.

## Continuous latent (advanced)

`VQGAE.encode` (called inside `encode_smiles` via `vqgae_encode_mols`) actually
returns three things: post-VQ atom vectors `(B, 51, 512)`, a continuous
molecule-level latent `(B, 512)`, and the discrete codebook indices we just
used. Only the discrete side is consumed by the public decode path. The
continuous path is the natural BO/RL search space — see tutorial 3 for what
exposing it would require.

## Next

- **`02_inverse_qsar_tubulin.ipynb`** — full inverse-QSAR pipeline: encode
  603 Tubulin compounds, train a Random Forest, run a Genetic Algorithm
  over fragment counts, decode hits.
- **`03_bring_your_own_optimizer.ipynb`** — same Tubulin objective, swap the
  GA for Optuna's TPE (discrete Bayesian optimization). Template for plugging
  in your own optimizer.
